# Day 9 — QLoRA fine-tuning benchmark (B7), Vendor Intelligence Copilot

Fine-tunes a small open model on this project's execution-verified question→SQL pairs
(`evals/datasets/sql_pairs_v1.jsonl`, generated by `scripts/generate_sql_pairs.py` — see
`docs/BUILD_LOG.md`'s Day 9 entries), then holds out 15% to compare against the base model
later in `eval_compare.py`.

**Before running:** Runtime → Change runtime type → T4 GPU (free tier is enough for a 1.5B
model in 4-bit + a LoRA adapter).

Read `finetuning/README.md` in the repo for the full picture (why this is a separate notebook,
why this base model, how to run the comparison afterward). This notebook only does the
fine-tuning half.

## 1. Install dependencies

Same list as `finetuning/requirements-colab.txt` — kept in sync by hand since Colab's own `!pip install -r` needs the file uploaded too; inlined here so this cell alone is enough to get started.

In [ ]:
!pip install -q "torch>=2.3.0" "transformers>=4.44.0" "peft>=0.12.0" \
    "bitsandbytes>=0.43.0" "trl>=0.9.6" "accelerate>=0.33.0" "datasets>=2.20.0" "duckdb>=1.0.0"

## 2. Upload the dataset

Upload two files from the repo:
- `evals/datasets/sql_pairs_v1.jsonl` (400 execution-verified question→SQL pairs)
- `data/snapshot/MANIFEST.json` (table/column names only — no data, licence-safe, already
  committed to the repo)

In [ ]:
from google.colab import files

print("Upload evals/datasets/sql_pairs_v1.jsonl :")
uploaded = files.upload()
PAIRS_PATH = next(iter(uploaded))

print("Upload data/snapshot/MANIFEST.json :")
uploaded2 = files.upload()
MANIFEST_PATH = next(iter(uploaded2))

## 3. Build the schema card and the prompt template

Same shape the production system's own `copilot.sql.schema.build_schema_card` renders (table
name, data class, columns) — reading the same `MANIFEST.json` the real app reads from, not a
hand-copied schema that could drift out of sync with it. Rebuilt here rather than imported
because this notebook is meant to run standalone in Colab without cloning the whole private
repo.

In [ ]:
import json

manifest = json.load(open(MANIFEST_PATH))
schema_lines = []
for table in sorted(manifest["tables"]):
    cols = [c for c in manifest["tables"][table]["columns"] if c != "MeetsMinK"]
    schema_lines.append(f"{table}: {', '.join(cols)}")
SCHEMA_CARD = "\n".join(schema_lines)
print(SCHEMA_CARD[:400], "...")


def build_prompt(question: str) -> str:
    return (
        "You are a careful SQL analyst. Given the schema below, write ONE DuckDB SELECT "
        "statement that answers the question. Only use the tables and columns shown. "
        "Respond with SQL only, no explanation.\n\n"
        f"### Schema\n{SCHEMA_CARD}\n\n"
        f"### Question\n{question}\n\n"
        "### SQL\n"
    )

## 4. Load the pairs, split train / held-out

85/15 split, fixed seed so re-running this notebook reproduces the same held-out set --
`eval_compare.py` needs the exact file this cell writes, not a fresh re-split, so the later
comparison is apples-to-apples against the model that was actually trained.

In [ ]:
import random

pairs = [json.loads(line) for line in open(PAIRS_PATH) if line.strip()]
print(f"loaded {len(pairs)} pairs")

rng = random.Random(42)
rng.shuffle(pairs)
split = int(len(pairs) * 0.85)
train_pairs, held_out_pairs = pairs[:split], pairs[split:]
print(f"train: {len(train_pairs)}  held-out: {len(held_out_pairs)}")

with open("held_out.jsonl", "w") as f:
    for p in held_out_pairs:
        f.write(json.dumps(p) + "\n")
print("wrote held_out.jsonl -- download this at the end, eval_compare.py needs it exactly")

## 5. Load the base model in 4-bit (QLoRA) and attach a LoRA adapter

`BASE_MODEL` — swap this if you want a different base; nothing below assumes Qwen specifically.
See `finetuning/README.md` for why this one was picked.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

BASE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, quantization_config=bnb_config, device_map="auto"
)
base_model = prepare_model_for_kbit_training(base_model)

# r=16, alpha=32 (2x r, a common default -- ADR-style note: this is the standard starting
# point LoRA's own paper and most QLoRA tutorials use, not tuned for this task specifically;
# a real hyperparameter sweep is explicitly out of scope for a Day-9 sprint benchmark).
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

## 6. Build completion-only-loss training examples

Tokenize `prompt + sql`, then mask every label at a prompt-token position to `-100` so the loss
only ever grades the SQL tokens — see `finetuning/README.md`'s "What completion-only loss means
here" section for why this specific masking approach.

In [ ]:
from datasets import Dataset

MAX_LEN = 512


def tokenize_example(example):
    prompt = build_prompt(example["question"])
    full_text = prompt + example["sql"] + tokenizer.eos_token

    full = tokenizer(full_text, truncation=True, max_length=MAX_LEN, padding="max_length")
    prompt_ids = tokenizer(prompt, truncation=True, max_length=MAX_LEN)["input_ids"]
    prompt_len = len(prompt_ids)

    labels = list(full["input_ids"])
    for i in range(min(prompt_len, len(labels))):
        labels[i] = -100
    # Also mask the right-padding the tokenizer added, so padding tokens never contribute loss.
    for i, attn in enumerate(full["attention_mask"]):
        if attn == 0:
            labels[i] = -100

    full["labels"] = labels
    return full


train_dataset = Dataset.from_list(train_pairs).map(tokenize_example)
train_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
print(train_dataset)

## 7. Train

Small, cheap-on-free-Colab settings (short run, small batch + gradient accumulation) --
deliberately not tuned for best possible accuracy; the point of this benchmark is an honest
base-vs-few-shot-vs-QLoRA comparison, not a state-of-the-art text-to-SQL model.

In [ ]:
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir="qlora_output",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    learning_rate=2e-4,
    logging_steps=10,
    save_strategy="no",
    bf16=True,
    report_to=[],
)

trainer = Trainer(model=model, args=training_args, train_dataset=train_dataset)
trainer.train()

## 8. Save the adapter and a quick sanity check

In [ ]:
ADAPTER_DIR = "qlora_adapter"
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f"adapter saved to {ADAPTER_DIR}/")

model.eval()
for example in held_out_pairs[:3]:
    prompt = build_prompt(example["question"])
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=120, do_sample=False)
    generated = tokenizer.decode(out[0][inputs["input_ids"].shape[1] :], skip_special_tokens=True)
    print("Q:", example["question"])
    print("ground truth SQL:", example["sql"])
    print("model generated :", generated.strip())
    print("---")

## 9. Download the artifacts

Zip the adapter, then download both files -- `eval_compare.py` (in `finetuning/` in the repo)
needs exactly these, by the exact names it expects.

In [ ]:
import shutil

shutil.make_archive("qlora_adapter", "zip", ADAPTER_DIR)
files.download("qlora_adapter.zip")
files.download("held_out.jsonl")
print("Also download data/snapshot/snapshot.duckdb from wherever you built the real snapshot --")
print("eval_compare.py executes every candidate SQL against it. See finetuning/README.md step 4.")